In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("order_id", "customer_id").orderBy(F.col("_commit_timestamp").desc())

    (
        microBatchDF.filter(F.col("_change_type").isin(["insert", "update_postimage"]))
            .withColumn("rank", F.rank().over(window))
            .filter("rank = 1")
            .drop("rank", "_change_type", "_commit_version")
            .withColumnRenamed("_commit_timestamp", "processed_timestamp")
            .createOrReplaceTempView("ranked_updates")
    )

    sql_query = """
        MERGE INTO dev.silver.customers_orders c
        USING ranked_updates r
        ON c.order_id = r.order_id AND c.customer_id = r.customer_id
        WHEN MATCHED AND c.processed_timestamp < r.processed_timestamp THEN
        UPDATE SET *
        WHEN NOT MATCHED THEN
        INSERT *
    """

    microBatchDF.sparkSession.sql(sql_query)

In [0]:
def process_customers_orders():
    orders_df = spark.readStream.table("dev.silver.orders_silver")

    cdf_customers_df = (
        spark.readStream
            .option("readChangeData", True)
            .option("startingVersion", 2)
            .table("dev.silver.customers_silver")
            .alias("c")
    )    

    sql_query = (
        orders_df.join(
            cdf_customers_df,
            orders_df.customer_id == cdf_customers_df.customer_id,
            "inner"
        )
        .drop(cdf_customers_df.customer_id)
        .writeStream
            .foreachBatch(batch_upsert)
            .option("checkpointLocation", "dbfs:/Volumes/dev/landing_zone/kafka_source/checkpoints/customers_orders")
            .trigger(availableNow=True)
            .start()        
    )
    sql_query.awaitTermination()
process_customers_orders()